In [1]:
import pandas as pd
from pathlib import Path

# Caminho base dos dados
DATA = Path("../../data")

# Função para carregar todos os CSVs de um prefixo
def load_csvs(prefix):
    files = sorted(DATA.glob(f"{prefix}-*.csv"))
    frames = []
    for file in files:
        df = pd.read_csv(file, sep=";", quotechar='"', low_memory=False)
        df["ano"] = int(file.stem.split("-")[-1])  # salva o ano como coluna
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

# Carregando os dados
votacoes         = load_csvs("voting/votacoes")
votos            = load_csvs("voting/votes/votacoesVotos")
votacoes_props   = load_csvs("voting/proposition/votacoesProposicoes")
temas_propos     = load_csvs("propositions/proposicoesTemas")
orientacoes      = load_csvs("voting/orientations/votacoesOrientacoes")
autores          = load_csvs("authors/proposicoesAutores")

deputados        = pd.read_csv("../../data/deputies/deputados.csv", sep=";", quotechar='"', low_memory=False)
legislaturas     = pd.read_csv("../../data/extra/legislaturas.csv", sep=";", quotechar='"', low_memory=False)
orgaos           = pd.read_csv("../../data/extra/orgaos.csv", sep=";", quotechar='"', low_memory=False)

# Prints para verificar estrutura dos dados
print("🔹 votacoes:")
print(votacoes.head(), "\n")

print("🔹 votos:")
print(votos.head(), "\n")

print("🔹 votacoes_props:")
print(votacoes_props.head(), "\n")

print("🔹 temas_propos:")
print(temas_propos.head(), "\n")

print("🔹 orientacoes:")
print(orientacoes.head(), "\n")

🔹 votacoes:
         id                                                uri        data  \
0  41577-14  https://dadosabertos.camara.leg.br/api/v2/vota...  2003-01-29   
1  96076-49  https://dadosabertos.camara.leg.br/api/v2/vota...  2003-02-19   
2  98922-29  https://dadosabertos.camara.leg.br/api/v2/vota...  2003-02-19   
3  102277-6  https://dadosabertos.camara.leg.br/api/v2/vota...  2003-02-19   
4  103255-3  https://dadosabertos.camara.leg.br/api/v2/vota...  2003-02-19   

  dataHoraRegistro  idOrgao  \
0              NaN        4   
1              NaN      180   
2              NaN      180   
3              NaN      180   
4              NaN      180   

                                            uriOrgao siglaOrgao  idEvento  \
0  https://dadosabertos.camara.leg.br/api/v2/orga...       MESA         0   
1  https://dadosabertos.camara.leg.br/api/v2/orga...       PLEN      3282   
2  https://dadosabertos.camara.leg.br/api/v2/orga...       PLEN      3282   
3  https://dadosabertos.

In [2]:
# Novo merge com sufixos personalizados
base = pd.merge(
    votacoes_props,
    votacoes,
    left_on='idVotacao',
    right_on='id',
    how='inner',
    suffixes=('_prop', '_vot')
)

# Verificando colunas disponíveis
print("📋 Colunas após merge:")
print(base.columns.tolist())

# Print das primeiras linhas com as colunas mais relevantes
print("\n🔎 Exemplo de linhas unidas:")
print(base[['idVotacao', 'proposicao_id', 'data_prop', 'descricao_prop', 'siglaOrgao', 'aprovacao']].head())

📋 Colunas após merge:
['idVotacao', 'uriVotacao', 'data_prop', 'descricao_prop', 'proposicao_id', 'proposicao_uri', 'proposicao_titulo', 'proposicao_ementa', 'proposicao_codTipo', 'proposicao_siglaTipo', 'proposicao_numero', 'proposicao_ano', 'ano_prop', 'id', 'uri', 'data_vot', 'dataHoraRegistro', 'idOrgao', 'uriOrgao', 'siglaOrgao', 'idEvento', 'uriEvento', 'aprovacao', 'votosSim', 'votosNao', 'votosOutros', 'descricao_vot', 'ultimaAberturaVotacao_dataHoraRegistro', 'ultimaAberturaVotacao_descricao', 'ultimaApresentacaoProposicao_dataHoraRegistro', 'ultimaApresentacaoProposicao_descricao', 'ultimaApresentacaoProposicao_idProposicao', 'ultimaApresentacaoProposicao_uriProposicao', 'ano_vot']

🔎 Exemplo de linhas unidas:
   idVotacao  proposicao_id   data_prop  \
0  100011-16         100011  2003-04-15   
1  100011-27         100011  2003-06-11   
2  100026-25         100026  2003-10-08   
3  100026-28         100026  2003-10-08   
4  100402-20         100402  2003-10-22   

           

In [3]:
# Mantemos apenas o primeiro tema de cada proposição (se houver)
temas_unicos = temas_propos.drop_duplicates(subset="uriProposicao", keep="first")

# Junta com a base
base = pd.merge(
    base,
    temas_unicos,
    left_on="proposicao_uri",
    right_on="uriProposicao",
    how="left"
)

print("🔎 Exemplo com tema:")
print(base[['proposicao_id', 'proposicao_ementa', 'tema', 'relevancia']].head())


🔎 Exemplo com tema:
   proposicao_id                                  proposicao_ementa  \
0         100011  Dispõe sobre a obrigatoriedade de as fábricas ...   
1         100011  Dispõe sobre a obrigatoriedade de as fábricas ...   
2         100026  Dispõe  sobre o trabalho escolar de estudantes...   
3         100026  Dispõe  sobre o trabalho escolar de estudantes...   
4         100402  Dispõe sobre a proibição de comercialização de...   

                             tema  relevancia  
0  Indústria, Comércio e Serviços         0.0  
1  Indústria, Comércio e Serviços         0.0  
2                        Educação         0.0  
3                        Educação         0.0  
4  Indústria, Comércio e Serviços         0.0  


In [4]:
print("📋 Colunas disponíveis no dataframe 'autores':")
print(autores.columns.tolist())

# Mostra as 5 primeiras linhas
print("\n🔎 Primeiras linhas de 'autores':")
print(autores.head())


📋 Colunas disponíveis no dataframe 'autores':
['idProposicao', 'uriProposicao', 'idDeputadoAutor', 'uriAutor', 'codTipoAutor', 'tipoAutor', 'nomeAutor', 'siglaPartidoAutor', 'uriPartidoAutor', 'siglaUFAutor', 'ordemAssinatura', 'proponente', 'ano']

🔎 Primeiras linhas de 'autores':
   idProposicao                                      uriProposicao  \
0       2122529  https://dadosabertos.camara.leg.br/api/v2/prop...   
1       1307620  https://dadosabertos.camara.leg.br/api/v2/prop...   
2       1307617  https://dadosabertos.camara.leg.br/api/v2/prop...   
3        955589  https://dadosabertos.camara.leg.br/api/v2/prop...   
4        955587  https://dadosabertos.camara.leg.br/api/v2/prop...   

   idDeputadoAutor                                           uriAutor  \
0              NaN  https://dadosabertos.camara.leg.br/api/v2/orga...   
1              NaN  https://dadosabertos.camara.leg.br/api/v2/orga...   
2          74353.0  https://dadosabertos.camara.leg.br/api/v2/depu...   
3   

In [5]:
# Número total de autores por proposição
autores_count = autores.groupby("idProposicao").size().reset_index(name="num_autores")

# Junta essa contagem com a base principal
base = pd.merge(base, autores_count, left_on="proposicao_id", right_on="idProposicao", how="left")

# Seleciona o autor principal (ordemAssinatura == 1)
autores_principal = autores[autores["ordemAssinatura"] == 1]

# Junta o autor principal na base
base = pd.merge(
    base,
    autores_principal[[
        "idProposicao", "nomeAutor", "siglaPartidoAutor", "siglaUFAutor", "tipoAutor"
    ]],
    left_on="proposicao_id",
    right_on="idProposicao",
    how="left"
)

# Print de verificação
print("🔎 Base com autoria enriquecida:")
print(base[['proposicao_id', 'nomeAutor', 'siglaPartidoAutor', 'siglaUFAutor', 'tipoAutor', 'num_autores']].head())



🔎 Base com autoria enriquecida:
   proposicao_id       nomeAutor siglaPartidoAutor siglaUFAutor    tipoAutor  \
0         100011     Neuton Lima               PFL           SP  Deputado(a)   
1         100011     Neuton Lima               PFL           SP  Deputado(a)   
2         100026  Jovair Arantes              PSDB           GO  Deputado(a)   
3         100026  Jovair Arantes              PSDB           GO  Deputado(a)   
4         100402      Cabo Júlio               PST           MG  Deputado(a)   

   num_autores  
0          1.0  
1          1.0  
2          1.0  
3          1.0  
4          1.0  


In [6]:
# Seleção das colunas úteis
df = base[[
    'proposicao_id',
    'proposicao_ementa',
    'siglaPartidoAutor',
    'siglaUFAutor',
    'tema',
    'tipoAutor',
    'num_autores',
    'aprovacao'
]].dropna(subset=['proposicao_ementa', 'siglaPartidoAutor', 'aprovacao'])

# Remove duplicatas por proposição
df = df.drop_duplicates(subset='proposicao_id')

# Verificação de balanceamento
print("🔢 Distribuição da variável alvo (aprovacao):")
print(df['aprovacao'].value_counts())

# Mostra uma amostra
print("\n🔎 Amostra da base de features:")
print(df.head())



🔢 Distribuição da variável alvo (aprovacao):
aprovacao
1.0    29005
0.0     1158
Name: count, dtype: int64

🔎 Amostra da base de features:
    proposicao_id                                  proposicao_ementa  \
0          100011  Dispõe sobre a obrigatoriedade de as fábricas ...   
2          100026  Dispõe  sobre o trabalho escolar de estudantes...   
4          100402  Dispõe sobre a proibição de comercialização de...   
5          100481  Altera o art. 149 do Decreto-Lei nº 2.848, de ...   
11         101021  Propõe que a Comissão de Defesa do Consumidor,...   

   siglaPartidoAutor siglaUFAutor  \
0                PFL           SP   
2               PSDB           GO   
4                PST           MG   
5                PFL           BA   
11               PFL           MA   

                                           tema                   tipoAutor  \
0                Indústria, Comércio e Serviços                 Deputado(a)   
2                                      Educação

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils import resample
import numpy as np

# ➤ 1. Separar classes
df_aprovado = df[df['aprovacao'] == 1]
df_rejeitado = df[df['aprovacao'] == 0]

# ➤ 2. Undersample da classe majoritária
df_aprovado_sampled = resample(df_aprovado, 
                               replace=False, 
                               n_samples=len(df_rejeitado), 
                               random_state=42)

df_balanced = pd.concat([df_aprovado_sampled, df_rejeitado])

# ➤ 3. Definir X e y
X = df_balanced.drop(columns=['aprovacao', 'proposicao_id'])
y = df_balanced['aprovacao']

# ➤ 4. Separar treino/teste
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

# ➤ 5. Definir colunas
categorical_cols = ['siglaPartidoAutor', 'siglaUFAutor', 'tema', 'tipoAutor']
text_col = 'proposicao_ementa'
numeric_col = ['num_autores']

# ➤ 6. Preprocessamento via pipeline
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('text', TfidfVectorizer(max_features=1000), text_col),
    ('num', 'passthrough', numeric_col)
])

# ➤ 7. Pipeline de modelo
clf = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=500))
])

# ➤ 8. Treinamento
clf.fit(X_train, y_train)

# ➤ 9. Avaliação
y_pred = clf.predict(X_test)
print("📊 Relatório de Classificação:")
print(classification_report(y_test, y_pred))


📊 Relatório de Classificação:
              precision    recall  f1-score   support

         0.0       0.61      0.70      0.66       348
         1.0       0.65      0.55      0.60       347

    accuracy                           0.63       695
   macro avg       0.63      0.63      0.63       695
weighted avg       0.63      0.63      0.63       695



In [8]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier

# Reutiliza o preprocessor da etapa anterior
modelos = {
    "LogReg": LogisticRegression(max_iter=500),
    "SVM": SVC(kernel="linear", probability=True),
    "DecisionTree": DecisionTreeClassifier(),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), activation="relu", max_iter=200),
    "KNN": KNeighborsClassifier(n_neighbors=9),
}

# Treinar e avaliar individualmente
for nome, modelo in modelos.items():
    pipe = Pipeline([
        ('preprocess', preprocessor),
        ('model', modelo)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    print(f"\n📊 {nome} — Relatório de Classificação:")
    print(classification_report(y_test, y_pred))



📊 LogReg — Relatório de Classificação:
              precision    recall  f1-score   support

         0.0       0.61      0.70      0.66       348
         1.0       0.65      0.55      0.60       347

    accuracy                           0.63       695
   macro avg       0.63      0.63      0.63       695
weighted avg       0.63      0.63      0.63       695


📊 SVM — Relatório de Classificação:
              precision    recall  f1-score   support

         0.0       0.61      0.69      0.65       348
         1.0       0.64      0.55      0.59       347

    accuracy                           0.62       695
   macro avg       0.62      0.62      0.62       695
weighted avg       0.62      0.62      0.62       695


📊 DecisionTree — Relatório de Classificação:
              precision    recall  f1-score   support

         0.0       0.60      0.59      0.59       348
         1.0       0.60      0.61      0.61       347

    accuracy                           0.60       695
   ma

e:\Anaconda\envs\dados_abertos\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



📊 MLP — Relatório de Classificação:
              precision    recall  f1-score   support

         0.0       0.59      0.63      0.61       348
         1.0       0.60      0.57      0.58       347

    accuracy                           0.60       695
   macro avg       0.60      0.60      0.60       695
weighted avg       0.60      0.60      0.60       695


📊 KNN — Relatório de Classificação:
              precision    recall  f1-score   support

         0.0       0.61      0.66      0.64       348
         1.0       0.63      0.58      0.60       347

    accuracy                           0.62       695
   macro avg       0.62      0.62      0.62       695
weighted avg       0.62      0.62      0.62       695



In [ ]:
# Usar apenas os 3 melhores modelos como no paper
ensemble = Pipeline([
    ('preprocess', preprocessor),
    ('model', VotingClassifier(
        estimators=[
            ('mlp', modelos['MLP']),
            ('svm', modelos['SVM']),
            ('knn', modelos['KNN']),
        ],
        voting='soft'
    ))  
])

ensemble.fit(X_train, y_train)
y_pred_ensemble = ensemble.predict(X_test)

print("\n🏆 Ensemble (Voting) — Relatório de Classificação:")
print(classification_report(y_test, y_pred_ensemble))


e:\Anaconda\envs\dados_abertos\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



🏆 Ensemble (Voting) — Relatório de Classificação:
              precision    recall  f1-score   support

         0.0       0.60      0.67      0.63       348
         1.0       0.63      0.56      0.59       347

    accuracy                           0.61       695
   macro avg       0.62      0.61      0.61       695
weighted avg       0.62      0.61      0.61       695

